In [1]:
#  item-based collaborative
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------
# 1) Load prepared interactions
# ----------------------------
# Expected columns: user_id, item_id, event_weight

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
PREP_ROOT = DATA_ROOT / "prepared"
FILE_PATH = PREP_ROOT / "interactions_prepared.csv"

df = pd.read_csv(FILE_PATH)
#df = pd.read_csv("../data/prepared/interactions_prepared.csv")

required_cols = {"user_id", "item_id", "event_weight"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Aggregate duplicate user-item interactions
df = (
    df.groupby(["user_id", "item_id"], as_index=False)["event_weight"]
      .sum()
)

# Keep users with at least 2 items so we can hold one out for test
user_counts = df.groupby("user_id")["item_id"].nunique()
eligible_users = user_counts[user_counts >= 2].index
df = df[df["user_id"].isin(eligible_users)].copy()

print("Shape after filtering:", df.shape)
print("Users:", df["user_id"].nunique(), "Items:", df["item_id"].nunique())

# ----------------------------
# 2) Leave-one-out split
# ----------------------------
# Hold out one item per user for evaluation
df = df.sample(frac=1, random_state=42).copy()
test_idx = df.groupby("user_id").head(1).index

test_df = df.loc[test_idx].copy()
train_df = df.drop(test_idx).copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# ----------------------------
# 3) Build train matrix
# ----------------------------
user_ids = train_df["user_id"].unique()
item_ids = train_df["item_id"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = {i: it for it, i in item_to_idx.items()}

train_df = train_df[train_df["item_id"].isin(item_to_idx)].copy()
test_df = test_df[test_df["item_id"].isin(item_to_idx)].copy()

rows = train_df["user_id"].map(user_to_idx)
cols = train_df["item_id"].map(item_to_idx)
vals = train_df["event_weight"].astype(float)

user_item = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_to_idx), len(item_to_idx))
)

# ----------------------------
# 4) Train item-item similarity
# ----------------------------
item_user = user_item.T
item_sim = cosine_similarity(item_user, dense_output=False)

print("User-item matrix:", user_item.shape)
print("Item-item similarity matrix:", item_sim.shape)

# ----------------------------
# 5) Recommendation function
# ----------------------------
def recommend_items(user_id, k=10):
    if user_id not in user_to_idx:
        return []

    uidx = user_to_idx[user_id]
    user_vector = user_item[uidx]                # 1 x n_items
    scores = user_vector.dot(item_sim).toarray().ravel()

    seen_items = set(train_df.loc[train_df["user_id"] == user_id, "item_id"].tolist())
    seen_indices = [item_to_idx[i] for i in seen_items if i in item_to_idx]
    scores[seen_indices] = -np.inf

    top_idx = np.argsort(scores)[::-1][:k]
    recs = [idx_to_item[i] for i in top_idx if np.isfinite(scores[i])]
    return recs

# Example
sample_user = train_df["user_id"].iloc[0]
print("Sample recommendations for user", sample_user, ":", recommend_items(sample_user, k=5))

# ----------------------------
# 6) Evaluate Precision@K, Recall@K, NDCG@K
# ----------------------------
def precision_recall_ndcg_at_k(test_df, k=10):
    user_truth = test_df.groupby("user_id")["item_id"].apply(set).to_dict()

    precisions, recalls, ndcgs = [], [], []

    for user_id, true_items in user_truth.items():
        if user_id not in user_to_idx:
            continue

        recs = recommend_items(user_id, k=k)
        if not recs:
            continue

        hits = [1 if item in true_items else 0 for item in recs]
        hit_count = sum(hits)

        precision = hit_count / k
        recall = hit_count / len(true_items)

        dcg = sum(hit / np.log2(idx + 2) for idx, hit in enumerate(hits))
        ideal_hits = [1] * min(len(true_items), k)
        idcg = sum(hit / np.log2(idx + 2) for idx, hit in enumerate(ideal_hits))
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f"Precision@{k}": np.mean(precisions) if precisions else 0.0,
        f"Recall@{k}": np.mean(recalls) if recalls else 0.0,
        f"NDCG@{k}": np.mean(ndcgs) if ndcgs else 0.0,
        "Evaluated Users": len(precisions)
    }

metrics_5 = precision_recall_ndcg_at_k(test_df, k=5)
metrics_10 = precision_recall_ndcg_at_k(test_df, k=10)

print("\nMetrics@5")
for k, v in metrics_5.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

print("\nMetrics@10")
for k, v in metrics_10.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


Shape after filtering: (1025679, 3)
Users: 288080 Items: 156487
Train shape: (737599, 3)
Test shape: (288080, 3)
User-item matrix: (288080, 132535)
Item-item similarity matrix: (132535, 132535)
Sample recommendations for user 684514 : [np.int64(428648), np.int64(420162), np.int64(374896), np.int64(251945), np.int64(1976)]

Metrics@5
Precision@5: 0.0105
Recall@5: 0.0527
NDCG@5: 0.0354
Evaluated Users: 260205

Metrics@10
Precision@10: 0.0076
Recall@10: 0.0756
NDCG@10: 0.0428
Evaluated Users: 260205
